# 📈 Notebook 4 — Visualisasi & Perbandingan Baseline
**Kelompok 1 — Tugas Besar Analisis Big Data**

Tahapan:
1. Perbandingan Random Forest vs Baseline (Decision Tree, SVM/Logistic)
2. Ringkasan metrik dalam tabel perbandingan
3. Visualisasi perbandingan
4. Kesimpulan akhir

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.classification import (
    RandomForestClassifier,
    DecisionTreeClassifier,
    LogisticRegression,
    GBTClassifier
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd

matplotlib.rcParams['figure.dpi'] = 120
pd.set_option('display.float_format', '{:.4f}'.format)

spark = SparkSession.builder \
    .appName('WaterPotability_Comparison') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

print('SparkSession berhasil dibuat.')

In [ ]:
train_df = spark.read.parquet('/home/jovyan/work/output/train_data.parquet')
test_df  = spark.read.parquet('/home/jovyan/work/output/test_data.parquet')
print(f'Training: {train_df.count():,} | Testing: {test_df.count():,}')

---
## 4.1 Training Semua Model Baseline

In [ ]:
import time

# Definisi semua model yang akan dibandingkan
models_config = {
    'Random Forest (Utama)': RandomForestClassifier(
        labelCol='Potability', featuresCol='features',
        numTrees=100, maxDepth=10, seed=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        labelCol='Potability', featuresCol='features',
        maxDepth=10, seed=42
    ),
    'Logistic Regression': LogisticRegression(
        labelCol='Potability', featuresCol='features',
        maxIter=100
    ),
    'Gradient Boosting': GBTClassifier(
        labelCol='Potability', featuresCol='features',
        maxIter=50, maxDepth=5, seed=42
    ),
}

evaluator_mc = MulticlassClassificationEvaluator(
    labelCol='Potability', predictionCol='prediction'
)

results = []

for model_name, model in models_config.items():
    print(f'\nTraining: {model_name}...', end=' ')
    start = time.time()
    fitted = model.fit(train_df)
    preds  = fitted.transform(test_df)
    elapsed = time.time() - start

    acc = evaluator_mc.evaluate(preds, {evaluator_mc.metricName: 'accuracy'})
    f1  = evaluator_mc.evaluate(preds, {evaluator_mc.metricName: 'f1'})
    pre = evaluator_mc.evaluate(preds, {evaluator_mc.metricName: 'weightedPrecision'})
    rec = evaluator_mc.evaluate(preds, {evaluator_mc.metricName: 'weightedRecall'})

    results.append({
        'Model': model_name,
        'Accuracy': acc,
        'F1-Score': f1,
        'Precision': pre,
        'Recall': rec,
        'Waktu (s)': round(elapsed, 1)
    })
    print(f'✅ Acc={acc:.4f} F1={f1:.4f} ({elapsed:.1f}s)')

results_df = pd.DataFrame(results)
print('\n✅ Semua model selesai dilatih!')

---
## 4.2 Tabel Perbandingan Metrik

In [ ]:
results_display = results_df.copy()
for col in ['Accuracy', 'F1-Score', 'Precision', 'Recall']:
    results_display[col] = results_display[col].apply(lambda x: f'{x:.4f} ({x*100:.2f}%)')

print('=== TABEL PERBANDINGAN SEMUA MODEL ===')
print(results_display.to_string(index=False))

best_model = results_df.loc[results_df['Accuracy'].idxmax()]
print(f'\n🏆 Model terbaik: {best_model["Model"]} dengan Accuracy = {best_model["Accuracy"]:.4f}')

---
## 4.3 Visualisasi Perbandingan Model

In [ ]:
metrics = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
model_names = results_df['Model'].tolist()

x = np.arange(len(metrics))
width = 0.18
colors_models = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']

fig, ax = plt.subplots(figsize=(14, 7))

for i, (model_name, color) in enumerate(zip(model_names, colors_models)):
    values = [results_df[results_df['Model'] == model_name][m].values[0] for m in metrics]
    offset = (i - len(model_names)/2 + 0.5) * width
    bars = ax.bar(x + offset, values, width, label=model_name, color=color,
                  alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, rotation=45)

ax.set_xlabel('Metrik Evaluasi', fontsize=12)
ax.set_ylabel('Nilai Metrik', fontsize=12)
ax.set_title('Perbandingan Performa Model — Water Potability Prediction', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.15)
ax.axhline(y=0.95, color='red', linestyle='--', linewidth=1, label='Target (0.95)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/home/jovyan/work/output/figures/perbandingan_model.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Gambar disimpan: output/figures/perbandingan_model.png')

---
## 4.4 Perbandingan dengan Nilai Baseline dari Literatur

In [ ]:
# Nilai dari literatur (Alomani et al., 2022 dan studi terkait)
baseline_literature = {
    'SMO-SVM (Literatur)': {'Accuracy': 0.95, 'F1-Score': 0.94, 'Precision': 0.94, 'Recall': 0.94},
    'KNN (Literatur)'    : {'Accuracy': 0.92, 'F1-Score': 0.91, 'Precision': 0.91, 'Recall': 0.91},
    'Decision Tree (Lit)': {'Accuracy': 0.93, 'F1-Score': 0.92, 'Precision': 0.92, 'Recall': 0.92},
}

print('=== PERBANDINGAN DENGAN NILAI BASELINE LITERATUR ===')
print(f'{"Model":<35} {"Accuracy":>10} {"F1-Score":>10} {"Precision":>10} {"Recall":>10}')
print('-' * 80)

# Model kita
rf_result = results_df[results_df['Model'] == 'Random Forest (Utama)'].iloc[0]
print(f'✨ {"Random Forest (Kelompok 1)":<33} {rf_result["Accuracy"]:>10.4f} {rf_result["F1-Score"]:>10.4f} {rf_result["Precision"]:>10.4f} {rf_result["Recall"]:>10.4f}')
print('-' * 80)

for model_lit, vals in baseline_literature.items():
    print(f'  {model_lit:<33} {vals["Accuracy"]:>10.4f} {vals["F1-Score"]:>10.4f} {vals["Precision"]:>10.4f} {vals["Recall"]:>10.4f}')

---
## 4.5 Kesimpulan Akhir

In [ ]:
rf_acc = results_df[results_df['Model'] == 'Random Forest (Utama)']['Accuracy'].values[0]
rf_f1  = results_df[results_df['Model'] == 'Random Forest (Utama)']['F1-Score'].values[0]

print('=' * 60)
print('               KESIMPULAN PROYEK')
print('=' * 60)
print(f'''
Model Random Forest Classifier berbasis PySpark berhasil
dilatih dan dievaluasi pada dataset Water Potability (3.276 sampel)
dengan 10 parameter fisikokimia sebagai fitur input.

Hasil evaluasi pada test set (20% data = ~655 sampel):
  • Accuracy  : {rf_acc:.4f} ({rf_acc*100:.2f}%)
  • F1-Score  : {rf_f1:.4f}  ({rf_f1*100:.2f}%)

Performa model {"MELAMPAUI" if rf_acc >= 0.95 else "MENDEKATI"} target ≥ 0.95 yang ditetapkan dalam proposal.

Fitur yang paling berpengaruh dalam prediksi dapat dilihat
pada grafik Feature Importance di Notebook 3.

Implikasi:
  → Model ini dapat digunakan sebagai alat bantu skrining
    kualitas air secara cepat, murah, dan skalabel.
  → Namun tetap perlu verifikasi laboratorium untuk
    keputusan kebijakan kesehatan publik.
''')
print('=' * 60)

spark.stop()
print('\n✅ Seluruh analisis selesai. SparkSession dihentikan.')